# 02 · Pilot runs (GPU)

Step 4 of the implementation plan. Scores the 2,000-row enriched pilot with each P2
configuration, then on CPU:

1. **Weighted metrics** (`metrics.py`): per-label AUC, PR-AUC, ECE, Brier, failure rate and
   coverage, with the pilot inclusion weights. AUC and PR-AUC rank on `logit_<label>`.
2. **Calibration** (`calibrate.py`): five-fold cross-fitted weighted Platt scaling on
   `logit_<label>` (never `logit(p)`), for the ECE after calibration, plus one fit on all rows.
3. **Prompt injection** (`configs/injection_set.csv`): 40 synthetic comments scored with and
   without an injected line; flip rates at 0.5.
4. **Comparison with TF-IDF** (`pilot_tfidf.parquet`, committed) and the memo's decision rule.
5. **Civil Comments shift check**: the committed 3,000-row shift sample scored with the stack's
   configuration; evaluated on CPU with `python -m src.teacher.shift evaluate`.

Runs on **Kaggle** or **Google Colab**; the notebook detects which.

**Kaggle:** Accelerator **GPU T4 x2** (vLLM uses one GPU), Internet **On**. Add Input → Your
Work → Notebooks → **00 prepare data**.

**Colab:** Runtime → Change runtime type → **T4 GPU**. The NVIDIA driver must be 580 or newer
(the GPU check cell prints it). Copy `wiki_train_raw.parquet` from the 00 notebook's output to
Google Drive and set `COLAB_TRAIN` below. The model weights are downloaded from Hugging Face at
the pinned revision; an `HF_TOKEN` in Colab Secrets (key icon) is optional, never in a cell.

**Output:** the `pilot/` folder (on Colab, also copied to `COLAB_OUT` in Drive). Commit its
contents to `Model-Pipeline/artifacts/pilot/`, except `civil_*`, which go to
`Model-Pipeline/artifacts/shift/`. Ids and scores only, no comment text.

In [ ]:
REPO_URL = "https://github.com/Nishaant-Soni/Distillate.git"
REPO_REF = "Nishaant/Teacher"     # use a commit hash once this notebook's code is on main
VLLM_VERSION = "0.31.0"           # the version the smoke test passed with
# (protocol, shots, text version), in this order; P1 is out of scope for the pilot.
# v1 is the original pilot; v2b is the teacher prompt after the obscene follow-up.
CONFIGS = [("p2", "zs", "v1"), ("p2", "fs", "v1"), ("p2", "zs", "v2b")]

# Colab only (ignored on Kaggle). Drive is mounted only if a path is under /content/drive.
# Without Drive: upload the parquet with the Files sidebar to /content and set COLAB_OUT = None;
# the output then downloads as pilot.zip at the end.
COLAB_TRAIN = "/content/drive/MyDrive/distillate/wiki_train_raw.parquet"
COLAB_OUT = "/content/drive/MyDrive/distillate/pilot"

In [ ]:
# Install vLLM before anything imports torch: vLLM pins its own torch version.
# Takes 3 to 10 minutes. The dependency-conflict warnings at the end are about Kaggle packages
# this notebook does not use (RAPIDS, Google Cloud) and can be ignored.
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", f"vllm=={VLLM_VERSION}"], check=True)

In [ ]:
from pathlib import Path

REPO_DIR = Path("/tmp/Distillate")   # outside /kaggle/working, so the repo is not saved as output
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO_DIR / "Model-Pipeline"))
print("code_commit:", CODE_COMMIT)

# Before anything imports transformers: Kaggle's torchaudio does not match vLLM's torch.
from src.teacher.compat import hide_torchaudio  # noqa: E402

hide_torchaudio()

In [ ]:
# The GPUs torch sees. vLLM's torch is a CUDA 13 build, which needs NVIDIA driver 580 or newer.
import torch

print(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,driver_version", "--format=csv,noheader"], text=True))
print("torch", torch.__version__, "| CUDA", torch.version.cuda)
assert torch.cuda.is_available(), "torch cannot see a GPU: check the accelerator and driver"

In [ ]:
import json
import os
import time

import pandas as pd

from src.teacher.config import load_config, model_version
from src.teacher.data import find_file
from src.teacher.pilot import fewshot_examples, pilot_truth

cfg = load_config()
SEED = cfg["seed"]
MODEL_VERSION = model_version(cfg)
PILOT_DIR = Path(cfg["paths"]["pilot_dir"])          # committed pilot ids, few-shot ids, TF-IDF
# Kaggle sets this in every notebook session; Colab also has a /kaggle/input folder.
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if ON_KAGGLE:
    OUT_DIR = Path("/kaggle/working/pilot")
    TRAIN_PATH = find_file("/kaggle/input", cfg["data"]["files"]["train"])
    MODEL_DIR = find_file("/kaggle/input", "weights_manifest.json").parent
    weights_manifest = json.loads((MODEL_DIR / "weights_manifest.json").read_text())
    assert weights_manifest["revision"] == cfg["model"]["revision"], weights_manifest["revision"]
else:   # Colab
    from google.colab import drive, userdata
    from huggingface_hub import snapshot_download

    if any(str(p).startswith("/content/drive/") for p in (COLAB_TRAIN, COLAB_OUT) if p):
        drive.mount("/content/drive")
    OUT_DIR = Path("/content/pilot")
    TRAIN_PATH = Path(COLAB_TRAIN)
    assert TRAIN_PATH.name == cfg["data"]["files"]["train"], TRAIN_PATH
    try:
        hf_token = userdata.get("HF_TOKEN")
    except Exception:   # no secret, or access not granted: the model is public
        hf_token = None
    # A commit hash as the revision downloads exactly that snapshot.
    MODEL_DIR = Path(snapshot_download(
        repo_id=cfg["model"]["model_id"], revision=cfg["model"]["revision"],
        local_dir="/content/model", token=hf_token,
        allow_patterns=["*.json", "*.safetensors", "*.txt", "LICENSE"]))
OUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_parquet(TRAIN_PATH)
pilot_ids = pd.read_parquet(PILOT_DIR / "pilot_ids.parquet")
truth = pilot_truth(train, pilot_ids)
pilot_rows = truth[["id"]].merge(train[["id", "text"]], on="id", validate="one_to_one")
fewshot = json.loads((PILOT_DIR / "fewshot_ids.json").read_text())
examples = fewshot_examples(train, fewshot["ids"])
assert not {i for ids in fewshot["ids"].values() for i in ids} & set(pilot_ids["id"])
print(len(pilot_rows), "pilot rows;", len(examples), "few-shot examples")
print("model:", MODEL_VERSION)

In [ ]:
from transformers import AutoTokenizer

from src.teacher.prompts import PromptBuilder
from src.teacher.scoring import answer_token_ids

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
yes_ids, no_ids = answer_token_ids(tokenizer)
builders = {}
for protocol, shots, text_version in CONFIGS:
    b = PromptBuilder.from_config(cfg, tokenizer, examples if shots == "fs" else None,
                                  protocol=protocol, shots=shots, text_version=text_version)
    builders[b.prompt_version] = b
    print(b.prompt_version, "| max prompt tokens:", b.max_prompt_tokens)

In [ ]:
import os

# Start vLLM's engine process with spawn, not fork: a forked process cannot use CUDA once this
# kernel has initialized it (this happened on Colab).
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import vllm  # noqa: E402
from vllm import LLM  # noqa: E402

from src.teacher.scoring import vllm_score_fn

start = time.perf_counter()
llm = LLM(
    model=str(MODEL_DIR),
    dtype=cfg["model"]["dtype"],
    max_model_len=cfg["model"]["max_model_len"],
    seed=SEED,
    gpu_memory_utilization=0.90,
    enable_prefix_caching=True,   # the system message and comment are shared by six probes
    generation_config="vllm",     # ignore the model's sampling defaults; we set our own
)
score_vllm = vllm_score_fn(llm, yes_ids, no_ids, logprobs=cfg["scoring"]["logprobs"])
results = {"code_commit": CODE_COMMIT, "model_version": MODEL_VERSION,
           "vllm_version": vllm.__version__,
           "vllm_load_s": round(time.perf_counter() - start, 1), "runs": {}}
print("vLLM", vllm.__version__, "loaded in", results["vllm_load_s"], "s")

## 1. Score the pilot and the injection set

In [ ]:
from src.teacher.injection import flip_rates, load_injection_set
from src.teacher.injection import texts as injection_texts
from src.teacher.pilot import save_run, summarize_run
from src.teacher.scoring import score_texts

injection_set = load_injection_set(cfg["paths"]["injection_set"])
inj_base, inj_injected = injection_texts(injection_set)
MIN_COVERAGE, MAX_RETRIES = cfg["scoring"]["min_coverage"], cfg["scoring"]["max_retries"]

summaries, injection_tables = {}, {}
for pv, builder in builders.items():
    start = time.perf_counter()
    pred, info = score_texts(pilot_rows["id"].tolist(), pilot_rows["text"].tolist(), builder,
                             score_vllm, MIN_COVERAGE, MAX_RETRIES)
    info["seconds"] = round(time.perf_counter() - start, 1)
    info["rows_per_s"] = round(len(pilot_rows) / info["seconds"], 2)
    pred["prompt_version"] = pd.Series([pv] * len(pred), dtype="string")
    pred["model_version"] = pd.Series([MODEL_VERSION] * len(pred), dtype="string")

    summary = summarize_run(pred, truth, pilot_ids, cfg, pv, MODEL_VERSION)
    save_run(OUT_DIR, pred, summary, info)
    summaries[pv] = summary

    inj_ids = injection_set["id"].tolist()
    base, _ = score_texts(inj_ids, inj_base, builder, score_vllm, MIN_COVERAGE, MAX_RETRIES)
    injected, _ = score_texts(inj_ids, inj_injected, builder, score_vllm, MIN_COVERAGE,
                              MAX_RETRIES)
    flips = flip_rates(injection_set, base, injected, cfg["injection"]["threshold"])
    flips.to_csv(OUT_DIR / f"injection_{pv}.csv", index=False)
    injection_tables[pv] = flips

    results["runs"][pv] = {"info": info, "saturation": summary["saturation"]}
    print(pv, info)
    print("  saturation:", summary["saturation"]["share_p_eq_1"], "of cells have p == 1.0")

## 2. Civil Comments shift check

The committed shift sample (`artifacts/shift/civil_shift_ids.parquet`, 3,000 rows of the pinned
Hugging Face Civil Comments validation split) scored with the stack's configuration
(`shift.prompt_version`). TF-IDF, calibration, and the stack are applied on CPU afterwards:
`python -m src.teacher.shift evaluate` with this file in `data/shift/`.

In [ ]:
from src.teacher.shift import download_civil, load_civil, shift_rows

SHIFT = cfg["shift"]
shift_ids = pd.read_parquet(Path(cfg["paths"]["shift_dir"]) / "civil_shift_ids.parquet")
civil = load_civil(download_civil(cfg, "/tmp/civil"), SHIFT["split"], SHIFT["threshold"])
shift_df = shift_rows(civil, shift_ids)
shift_builder = builders[SHIFT["prompt_version"]]

start = time.perf_counter()
civil_pred, civil_info = score_texts(shift_df["id"].tolist(), shift_df["text"].tolist(),
                                     shift_builder, score_vllm, MIN_COVERAGE, MAX_RETRIES)
civil_info.update(seconds=round(time.perf_counter() - start, 1), code_commit=CODE_COMMIT)
civil_pred["prompt_version"] = pd.Series([shift_builder.prompt_version] * len(civil_pred),
                                         dtype="string")
civil_pred["model_version"] = pd.Series([MODEL_VERSION] * len(civil_pred), dtype="string")
civil_pred.to_parquet(OUT_DIR / f"civil_{shift_builder.prompt_version}.parquet", index=False)
(OUT_DIR / f"civil_info_{shift_builder.prompt_version}.json").write_text(
    json.dumps(civil_info, indent=2) + "\n")
print(civil_info)

## 3. Results

In [ ]:
from src.teacher.metrics import evaluate
from src.teacher.pilot import decision, results_table

tfidf_name = cfg["baseline"]["model_version"]
tfidf = evaluate(pd.read_parquet(PILOT_DIR / "pilot_tfidf.parquet"), truth, pilot_ids,
                 cfg["metrics"]["ece_bins"])
speeds = {pv: r["info"]["rows_per_s"] for pv, r in results["runs"].items()}
table = results_table(summaries, tfidf, tfidf_name, speeds)
results["decision"] = decision(table, tfidf_name)
results["table"] = json.loads(table.round(5).to_json(orient="index"))
results["injection"] = {pv: t[t["label"] == "all"].round(4).to_dict(orient="records")
                        for pv, t in injection_tables.items()}
(OUT_DIR / "pilot_results.json").write_text(json.dumps(results, indent=2) + "\n")

summary_cols = ["mean_auc", "mean_pr_auc", "ece_raw", "ece_calibrated", "brier_calibrated",
                "failure_rate", "coverage", "rows_per_s"]
display(table[summary_cols].round(4))
display(table[[c for c in table.columns if c.startswith("auc_")]].round(4))
print(json.dumps(results["decision"], indent=2))

In [ ]:
# Injection flip rates (pooled over labels) and per-label detail.
for pv, flips in injection_tables.items():
    print(pv)
    display(flips.round(3))

In [ ]:
# Accepted when the table has both P2 runs and TF-IDF and the failure rate is <= 1%
# (pilot runs and the shift sample).
failure = {pv: r["info"]["failed_rows"] / len(pilot_rows) for pv, r in results["runs"].items()}
failure["civil"] = civil_info["failed_rows"] / len(shift_df)
assert set(builders) | {tfidf_name} <= set(table.index), table.index
assert all(f <= 0.01 for f in failure.values()), f"failure rate above 1%: {failure}"
if not ON_KAGGLE:   # the Colab VM is deleted when the session ends
    import shutil

    if COLAB_OUT:
        shutil.copytree(OUT_DIR, COLAB_OUT, dirs_exist_ok=True)
        print("Copied to", COLAB_OUT)
    else:
        from google.colab import files

        files.download(shutil.make_archive("/content/pilot", "zip", OUT_DIR))
print("Pilot done. Download the output folder:", sorted(p.name for p in OUT_DIR.iterdir()))